# 03 — Run experiments

Configure and launch experiments from a **single configuration cell** below.

Each run writes under the configured storage root (`outputs/runs/` locally, `UH_CV/runs/` on Google Drive).

**Prerequisite:** notebook 02 → `data/annotations/validation_lengths.csv`

In [1]:
from src.colab_bootstrap import setup_notebook_environment
from src.config import get_config
from src.experiments import load_validation_image_ids
from src.experiments.notebook_helpers import (
    RunExperimentsConfig,
    build_experiment_specs,
    preview_experiment_specs,
    project_config_for_experiments,
    resolve_runs_root,
    run_configured_experiments,
)
from src.utils import setup_logging

REPO, STORAGE = setup_notebook_environment(mount_drive=True)
setup_logging()
cfg = get_config()
print(f"Storage: {STORAGE} | Colab: {cfg.is_colab}")
VALIDATION_IDS = load_validation_image_ids(cfg)
print(f"Validation set: {len(VALIDATION_IDS)} images")

Storage: /Users/sebastianinouye/Desktop/Everything/Projects/Github/fishnet_cv | Colab: False
2026-06-02 23:12:33 | INFO     | src.experiments | Loaded 30 validation image IDs from validation_lengths.csv
Validation set: 30 images


## Configuration

Edit this cell only. Set `RUN = True` to execute; leave `False` to preview the experiment grid without running.

Use `experiments=[...]` for explicit runs (legacy mode). Otherwise a grid is built from `pipelines` × `methods` × `splits`.

In [2]:
RUN = True  # set False to preview specs without executing

RUN_CFG = RunExperimentsConfig(
    # --- scope ---
    pipelines=["advanced"],           # "baseline" | "advanced"
    methods=["skeleton"],
    splits=["valid"],
    image_ids=None,                    # explicit list overrides validation_set_only
    validation_set_only=True,          # use all IDs from validation_lengths.csv
    limit=None,

    # --- execution ---
    overwrite=True,
    visualize=True,                   # save debug figures under run_dir/figures/
    evaluate=True,
    runs_root=None,                   # default: cfg.runs_root (Drive or outputs/runs)
    run_name_template="{pipeline}_{method}_grid_v1",

    # --- Colab / Drive persistence (all default True) ---
    cache_results=True,
    save_figures_to_drive=True,
    save_metrics_to_drive=True,
    save_predictions_to_drive=True,

    # --- advanced pipeline flags (when pipeline="advanced") ---
    perspective=False,
    use_grid_auto_calibration=True,
    use_depth_estimation=False,
    use_3d_measurement=False,

    # --- regression calibration (notebook 02 labels required) ---
    run_regression_calibration=False,   # True: train + evaluate sklearn length model
    regression_run_name="regression_calibrated_v1",
    regression_model_type="random_forest",  # or "xgboost" if installed
    regression_train_split="valid",

    # --- optional: explicit experiment list (overrides grid above) ---
    experiments=None,
    # experiments=[
    #     {"pipeline": "advanced", "method": "skeleton", "run_name": "advanced_full_v2",
    #      "split": "valid", "validation_set_only": True, "overwrite": True,
    #      "use_grid_auto_calibration": True, "use_depth_estimation": True,
    #      "use_3d_measurement": True},
    # ],
)

SPECS = build_experiment_specs(RUN_CFG)
preview_experiment_specs(SPECS, RUN_CFG)

,run_name,pipeline,method,split,visualize,perspective,grid,depth,3d
0,advanced_skeleton_grid_v1,advanced,skeleton,valid,True,False,True,False,False


## Run experiments

In [3]:
import pandas as pd

exp_cfg = project_config_for_experiments(RUN_CFG)
runs_root = resolve_runs_root(RUN_CFG)

if RUN:
    summary = run_configured_experiments(RUN_CFG, SPECS, cfg=exp_cfg)
else:
    print("RUN=False — skipped execution. Set RUN=True in the configuration cell.")
    summary = pd.DataFrame()

if len(summary):
    display(summary[[c for c in ["run_name", "pipeline", "method", "n_predictions", "mae_mm", "rmse_mm", "run_dir"] if c in summary.columns]])

2026-06-02 23:12:47 | INFO     | src.experiments | Running experiment 1/1: {'pipeline': 'advanced', 'method': 'skeleton', 'split': 'valid', 'run_name': 'advanced_skeleton_grid_v1', 'overwrite': True, 'visualize': True, 'evaluate': True, 'limit': None, 'validation_set_only': True, 'perspective': False, 'use_grid_auto_calibration': True, 'use_depth_estimation': False, 'use_3d_measurement': False}
2026-06-02 23:12:47 | INFO     | src.experiments | Loaded 30 validation image IDs from validation_lengths.csv
2026-06-02 23:12:47 | INFO     | src.experiments.run_manager | Saved run config: /Users/sebastianinouye/Desktop/Everything/Projects/Github/fishnet_cv/outputs/runs/advanced_skeleton_grid_v1/config.json
2026-06-02 23:12:47 | INFO     | src.pipelines.advanced | AdvancedPipeline: split=valid method=skeleton grid=True depth=False 3d=False perspective=False


Advanced (valid, n=30): 0img [00:00, ?img/s]

2026-06-02 23:12:47 | WARNING  | src.calibration.grid_auto | Grid calibration unreliable (conf=0.46, spacing=60.00); markers should be used
2026-06-02 23:12:47 | INFO     | src.calibration.marker | Estimated scale: 5.5492 px/mm (n=2 markers, mean)
2026-06-02 23:12:47 | WARNING  | src.pipelines.advanced_inference | 10786: grid calibration failed; falling back to marker scale
2026-06-02 23:12:51 | INFO     | src.evaluation | Loaded 30 ground-truth rows from /Users/sebastianinouye/Desktop/Everything/Projects/Github/fishnet_cv/data/annotations/validation_lengths.csv
2026-06-02 23:12:51 | WARNING  | src.calibration.grid_auto | Grid calibration unreliable (conf=0.46, spacing=60.00); markers should be used
2026-06-02 23:12:51 | INFO     | src.calibration.marker | Estimated scale: 5.5492 px/mm (n=2 markers, mean)
2026-06-02 23:12:51 | WARNING  | src.pipelines.advanced_inference | 10786: grid calibration failed; falling back to marker scale
2026-06-02 23:13:10 | INFO     | src.visualization._co

Advanced (valid, n=30): 1img [00:22, 22.86s/img]

2026-06-02 23:13:10 | WARNING  | src.calibration.grid_auto | Grid calibration unreliable (conf=0.48, spacing=88.00); markers should be used
2026-06-02 23:13:10 | INFO     | src.calibration.marker | Estimated scale: 2.5409 px/mm (n=4 markers, mean)
2026-06-02 23:13:10 | WARNING  | src.pipelines.advanced_inference | 11050: grid calibration failed; falling back to marker scale
2026-06-02 23:13:11 | INFO     | src.evaluation | Loaded 30 ground-truth rows from /Users/sebastianinouye/Desktop/Everything/Projects/Github/fishnet_cv/data/annotations/validation_lengths.csv
2026-06-02 23:13:11 | WARNING  | src.calibration.grid_auto | Grid calibration unreliable (conf=0.48, spacing=88.00); markers should be used
2026-06-02 23:13:11 | INFO     | src.calibration.marker | Estimated scale: 2.5409 px/mm (n=4 markers, mean)
2026-06-02 23:13:11 | WARNING  | src.pipelines.advanced_inference | 11050: grid calibration failed; falling back to marker scale
2026-06-02 23:13:16 | INFO     | src.visualization._co

...

Advanced (valid, n=30): 29img [06:48,  8.51s/img]

2026-06-02 23:19:35 | WARNING  | src.calibration.grid_auto | Grid calibration unreliable (conf=0.47, spacing=24.00); markers should be used
2026-06-02 23:19:35 | INFO     | src.calibration.marker | Estimated scale: 1.5060 px/mm (n=4 markers, mean)
2026-06-02 23:19:35 | WARNING  | src.pipelines.advanced_inference | 9701: grid calibration failed; falling back to marker scale
2026-06-02 23:19:36 | INFO     | src.evaluation | Loaded 30 ground-truth rows from /Users/sebastianinouye/Desktop/Everything/Projects/Github/fishnet_cv/data/annotations/validation_lengths.csv
2026-06-02 23:19:36 | WARNING  | src.calibration.grid_auto | Grid calibration unreliable (conf=0.47, spacing=24.00); markers should be used
2026-06-02 23:19:36 | INFO     | src.calibration.marker | Estimated scale: 1.5060 px/mm (n=4 markers, mean)
2026-06-02 23:19:36 | WARNING  | src.pipelines.advanced_inference | 9701: grid calibration failed; falling back to marker scale
2026-06-02 23:19:37 | INFO     | src.visualization._comm

Advanced (valid, n=30): 30img [06:50, 13.68s/img]

2026-06-02 23:19:37 | INFO     | src.pipelines.advanced_inference | Wrote 30 advanced predictions to /Users/sebastianinouye/Desktop/Everything/Projects/Github/fishnet_cv/outputs/runs/advanced_skeleton_grid_v1/predictions.csv
2026-06-02 23:19:37 | INFO     | src.evaluation | Loaded 30 ground-truth rows from /Users/sebastianinouye/Desktop/Everything/Projects/Github/fishnet_cv/data/annotations/validation_lengths.csv
2026-06-02 23:19:37 | INFO     | src.evaluation | Metrics (n=30): MAE=51.651 mm, RMSE=125.717 mm
2026-06-02 23:19:37 | INFO     | src.evaluation | Wrote comparison table to /Users/sebastianinouye/Desktop/Everything/Projects/Github/fishnet_cv/outputs/runs/advanced_skeleton_grid_v1/comparison.csv
2026-06-02 23:19:37 | INFO     | src.evaluation | Wrote metrics to /Users/sebastianinouye/Desktop/Everything/Projects/Github/fishnet_cv/outputs/runs/advanced_skeleton_grid_v1/metrics.json
2026-06-02 23:19:37 | INFO     | src.evaluation | Logged experiment to /Users/sebastianinouye/Deskt

,run_name,pipeline,method,n_predictions,mae_mm,rmse_mm,run_dir
0,advanced_skeleton_grid_v1,advanced,skeleton,30,51.651321,125.716694,/Users/sebastianinouye/Desktop/Everything/Proj...


## Sanity check

Confirms every annotated fish was predicted and evaluated when using the validation set.

In [4]:
if len(summary) and RUN_CFG.validation_set_only and RUN_CFG.image_ids is None:
    n_gt = len(VALIDATION_IDS)
    for _, row in summary.iterrows():
        pred_path = Path(row["predictions_path"])
        comp_path = Path(row["comparison_path"]) if pd.notna(row.get("comparison_path")) else None
        pred = pd.read_csv(pred_path)
        comp = pd.read_csv(comp_path) if comp_path and comp_path.is_file() else None
        print(f"{row['run_name']}: predictions={len(pred)}", end="")
        if comp is not None:
            print(f"  evaluated={len(comp)}", end="")
            assert len(pred) == n_gt, f"{row['run_name']}: not all validation IDs predicted"
            assert len(comp) == n_gt, f"{row['run_name']}: evaluation incomplete"
            print("  OK")
        else:
            print("  (no comparison.csv)")
else:
    print("Skipped — custom image_ids or no runs executed.")

NameError: name 'Path' is not defined